In [30]:
import pandas as pd
import matplotlib.pyplot as plt
from surprise import Reader, Dataset
from surprise import SVD
from surprise.model_selection import GridSearchCV
from surprise import SVDpp, NMF

In [16]:
df_ml = pd.read_csv('ml-latest-small/ratings.csv')
df_ml.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [17]:
df_ml = df_ml.drop(columns=['timestamp'])

In [18]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ml, reader)

In [23]:
param_grid = {
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

print("Найкраще значення RMSE:", gs.best_score['rmse'])
print("Найкращі параметри:", gs.best_params['rmse'])

Найкраще значення RMSE: 0.8759370385661099
Найкращі параметри: {'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.1}


In [28]:
param_grid_svdpp = {
    'n_epochs': [10, 19],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.09]
}

gs_svdpp = GridSearchCV(SVDpp, param_grid_svdpp, measures=['rmse', 'mae'], cv=3)
gs_svdpp.fit(data)

print("Найкраще значення RMSE (SVD++):", gs_svdpp.best_score['rmse'])
print("Найкращі параметри SVD++:", gs_svdpp.best_params['rmse'])

Найкраще значення RMSE (SVD++): 0.8696820581158398
Найкращі параметри SVD++: {'n_epochs': 19, 'lr_all': 0.005, 'reg_all': 0.02}


In [29]:
param_grid_nmf = {
    'n_epochs': [10, 20],
    'n_factors': [11, 15]
}

gs_nmf = GridSearchCV(NMF, param_grid_nmf, measures=['rmse', 'mae'], cv=3)
gs_nmf.fit(data)

print("Найкраще значення RMSE (NMF):", gs_nmf.best_score['rmse'])
print("Найкращі параметри NMF:", gs_nmf.best_params['rmse'])

Найкраще значення RMSE (NMF): 0.9329379464575962
Найкращі параметри NMF: {'n_epochs': 20, 'n_factors': 15}


Найкраще себе показав SVD++. У нього найменша помилка (RMSE ~0.869). Рахував він, правда, довго (21 хвилину), але якщо потрібна максимальна точність, то він найркаще підійде

Звичайний SVD теж добре: відпрацював за секунди, а результат видав майже такий самий (0.875). Це ідеальний варіант, коли треба швидко і без особливої втрати якості.

А от NMF на цих даних взагалі не зайшов, у нього помилка найбільша (0.932)

Томувибір для цього датасету - SVD++, якщо є час почекати на навчання, або базовий SVD для швидкості.